# Composing local causal models: reproducible experiments

Run B1–B13 in order from the supplementary-material directory. See README.md for installation, command-line reproduction, paper outputs, generation settings, and runtime. The experiments check supplied finite causal descriptions against known equations and independent oracles.


## B1 — Imports

Load the released helpers and keep runtime caches local. Use normal Python execution; correctness checks use assertions.


In [ ]:
import os
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "causal_experiments").is_dir(), "Start Jupyter from the project folder."
for variable, folder in [("MPLCONFIGDIR", "matplotlib"), ("IPYTHONDIR", "ipython")]:
    path = ROOT / ".runtime" / folder
    path.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(path)

import matplotlib.pyplot as plt
from IPython.display import display
from causal_experiments.experiments import (
    RunConfig, start_run, build_small_collection, run_observational_checks,
    run_observational_distributions, run_intervention_checks, run_running_example,
    run_diagnostics, run_duplication_comparison, run_structural_benchmarks,
    run_supporting_validation, finalize_run,
)
from causal_experiments.plots import plot_running_example, plot_scaling
from causal_experiments.artifacts import write_paper_tables, record_presentation

## B2 — Experiment configuration

The full profile uses 90 Boolean ground SCMs and four decompositions per model, producing 360 diagrams, plus 81 structural configurations. The smoke profile exercises the same procedures on a smaller grid. Set `CATEGORY_PROFILE=smoke` before starting Jupyter for a smoke run.


In [ ]:
PROFILE = os.environ.get("CATEGORY_PROFILE", "full")
assert PROFILE in {"full", "smoke"}
config = RunConfig(
    profile=PROFILE,
    small_sizes=[8, 16, 32] if PROFILE == "full" else [8],
    small_modules=[2, 4] if PROFILE == "full" else [2],
    small_copies=[0.0, 0.25],
    small_families=["chain", "layered", "random_acyclic"],
    small_seeds=list(range(10)) if PROFILE == "full" else [0],
    scale_sizes=[100, 1000, 10000] if PROFILE == "full" else [100],
    scale_modules=[2, 4, 8] if PROFILE == "full" else [2],
    scale_copies=[0.0, 0.25, 0.5],
    scale_seeds=[0, 1, 2] if PROFILE == "full" else [0],
    timing_repetitions=5,
    write_datasets=True,
    measure_memory=True,
)
print("Planned small diagrams:", len(config.small_grid))
print("Planned assignment/intervention scenarios:", sum(16 * (1 + 2 * c[0]) for c in config.small_grid))
print("Planned structural configurations:", len(config.scale_grid))

## B3 — Start the run

Record the configuration, software, available hardware details, and source hashes. The manifest remains incomplete until all required checks finish.


In [ ]:
run = start_run(ROOT, config, output_dir=os.environ.get("CATEGORY_OUTPUT_DIR"))
print("Output directory:", run.out)
print("Manifest status:", run.manifest["status"])

## B4 — Reconstruction

Generate faithfully fragmented models, check four-table recovery, verify certificates, and compare quotient partitions with an independent oracle.


In [ ]:
instances, correctness = build_small_collection(run)
display(correctness.groupby(["family", "n_endogenous"]).agg(
    diagrams=("instance", "count"),
    raw_reconstructed=("raw_reconstruction", "sum"),
    deduplicated_reconstructed=("variable_only_DE_reconstruction", "sum"),
))

## B5 — Observational evaluations

Enumerate every exogenous assignment. Compare global and induced-local evaluations with the original ordered ground equations.


In [ ]:
observational = run_observational_checks(run, instances)
display(observational.groupby("route").agg(
    assignment_checks=("assignments", "sum"),
    global_mismatches=("global_mismatches", "sum"),
    local_mismatches=("local_mismatches", "sum"),
    max_coordinate_error=("max_coordinate_error", "max"),
))

## B6 — Interventions and exact distributions

Check every singleton Boolean intervention through both assembly routes. Compare intervention before and after assembly. Accumulate exact probability masses under uniform, non-uniform product, and correlated laws.


In [ ]:
observational_laws = run_observational_distributions(run, instances)
interventions, distributions, correctness = run_intervention_checks(
    run, instances, correctness, observational_laws,
)
display(interventions.groupby("route").agg(
    intervention_cases=("target", "count"),
    assignment_checks=("assignments", "sum"),
    global_mismatches=("global_mismatches", "sum"),
    local_mismatches=("local_mismatches", "sum"),
    commutation_mismatches=("commutation_mismatches", "sum"),
))
display(distributions.groupby("law").agg(
    law_route_cases=("target", "count"),
    largest_global_TV=("global_TV_exact", "max"),
    largest_local_TV=("max_local_TV_exact", "max"),
))
print("Unique diagram/assignment/intervention scenarios:", int(correctness.all_scenarios.sum()))

## B7 — Running-example figure

Compute the running example and its exact intervention responses, then generate the corresponding figure from saved CSVs.


In [ ]:
response = run_running_example(run)
display(response)
figure = plot_running_example(run.out)
display(figure)
plt.close(figure)

## B8 — Diagnostic outcomes

Check fixture classifications and independently verify witnesses. F01–F15 are the primary diagnostics; F16–F19 are additional implementation controls.


In [ ]:
diagnostics, fixture_suite, fixture_assemblies = run_diagnostics(run)
primary = diagnostics[diagnostics.scope == "primary"]
diagnostic_table = primary.pivot(index="fixture", columns="mode", values="actual_valid")
diagnostic_table = diagnostic_table.join(primary.groupby("fixture").notes.first())
display(diagnostic_table)

## B9 — Paired duplicate comparison

Run both assembly methods on identical variable-only interfaces. Test no copies, identical copies, and copied equations with conflicting codes.


In [ ]:
duplication_cases, duplication_summary = run_duplication_comparison(run, instances)
display(duplication_summary)
print("Paired method cases:", len(duplication_cases))

## B10 — Structural benchmarks

Use one warm-up and five timed repetitions per configuration, with correctness checks outside the timed stages. Measure traced Python allocations in separate runs.


In [ ]:
timings, memory, timing_summary = run_structural_benchmarks(run)
display(timing_summary)
print("Completed structural configurations:", timings.instance.nunique())
print("Completed timed repetitions:", len(timings))
print("Completed separate allocation runs:", len(memory))

## B11 — Runtime, stage, and allocation figures

Regenerate the figures from measured CSVs. Runtime error bars show interquartile ranges across seeds/repetitions; allocation error bars show minimum–maximum seed ranges.


In [ ]:
figures = plot_scaling(run.out)
for figure in figures:
    display(figure)
    plt.close(figure)

## B12 — Supporting validation

Check repeated arguments, ordered inputs, typed outputs, simultaneous interventions, and deliberately corrupted certificates and records.


In [ ]:
supporting = run_supporting_validation(run, fixture_suite, fixture_assemblies)
display(diagnostics[diagnostics.scope == "supporting"][["fixture", "mode", "actual_valid", "notes"]])
print("Supporting implementation controls passed:", int(supporting.passed.sum()))

## B13 — Export table summaries and complete the manifest

Rebuild the empirical table values, verify completed checks and source hashes, and mark the run complete.


In [ ]:
write_paper_tables(run.out)
summary = finalize_run(
    run, correctness, observational, interventions, distributions,
    diagnostics, duplication_cases, timings, memory, supporting,
)
record_presentation(run.out)
display(summary)
print("Manifest status:", run.manifest["status"])

Restart the kernel before rerunning after helper changes. A rerun overwrites the chosen output directory; use `CATEGORY_OUTPUT_DIR` to write to a different directory.
